# ⚙️ Notebook 08: Ingeniería de Características — Churn

**Objetivo:** Seleccionar las features más discriminativas para predecir Churn usando métodos estadísticos y basados en modelos.

**Métodos:**
1. Variance Threshold
2. Mutual Information (Classification)
3. Prueba Chi-cuadrado (SelectKBest)
4. Importancia Random Forest
5. Coeficientes Logistic Regression (L1)
6. Ranking combinado y selección final

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import (
    SelectKBest, chi2, mutual_info_classif, VarianceThreshold
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import joblib, json, warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
np.random.seed(42)
print('✅ Librerías importadas')

## 1. Carga de Datos

In [ ]:
X_train    = pd.read_csv('../data/processed/churn_X_train.csv')
X_test     = pd.read_csv('../data/processed/churn_X_test.csv')
X_train_sc = pd.read_csv('../data/processed/churn_X_train_scaled.csv')
X_test_sc  = pd.read_csv('../data/processed/churn_X_test_scaled.csv')
y_train    = pd.read_csv('../data/processed/churn_y_train.csv').squeeze()
y_test     = pd.read_csv('../data/processed/churn_y_test.csv').squeeze()

with open('../data/processed/churn_feature_info.json') as f:
    fi = json.load(f)

print(f'📦 X_train: {X_train.shape} | Churn rate: {y_train.mean()*100:.1f}%')
print(f'📋 Total features: {len(fi["all_features"])}')

## 2. Variance Threshold

In [ ]:
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_train)
variances = pd.Series(vt.variances_, index=X_train.columns).sort_values()

low_var = variances[variances < 0.01].index.tolist()
print(f'⚠️ Features con varianza < 0.01: {low_var if low_var else "Ninguna"}')

fig, ax = plt.subplots(figsize=(14, 5))
variances.sort_values(ascending=False).plot(kind='bar', ax=ax, color='#3498db', edgecolor='white')
ax.axhline(y=0.01, color='red', linestyle='--', label='Umbral')
ax.set_title('Varianza por Feature', fontweight='bold')
ax.legend()
plt.xticks(rotation=60, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('../data/processed/churn_variance.png', dpi=150)
plt.show()

## 3. Mutual Information (Clasificación)

In [ ]:
# Mutual Information — captura relaciones no lineales
# Asegurarse que no hay negativos
X_train_nn = X_train.clip(lower=0)

mi_scores = mutual_info_classif(X_train_nn, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

print('📊 TOP 15 MUTUAL INFORMATION SCORES:')
print(mi_series.head(15).round(4).to_string())

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#e74c3c' if v > mi_series.median() else '#95a5a6' for v in mi_series.values]
mi_series.plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
ax.set_title('Mutual Information — Clasificación Churn', fontsize=13, fontweight='bold')
ax.axvline(x=mi_series.median(), color='navy', linestyle='--',
           label=f'Mediana: {mi_series.median():.4f}')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/churn_mi_scores.png', dpi=150)
plt.show()

## 4. Chi-Cuadrado (SelectKBest)

In [ ]:
# Chi2 requiere valores no negativos
X_train_abs = np.abs(X_train)

skb = SelectKBest(score_func=chi2, k='all')
skb.fit(X_train_abs, y_train)

chi2_scores = pd.DataFrame({
    'Chi2_Score': skb.scores_,
    'P_Value': skb.pvalues_
}, index=X_train.columns).sort_values('Chi2_Score', ascending=False)

chi2_scores['Significativa'] = chi2_scores['P_Value'] < 0.05

print('📊 TOP 15 CHI2 SCORES:')
print(chi2_scores.head(15).round(4).to_string())
print(f'\n✅ Features significativas: {chi2_scores["Significativa"].sum()}')

## 5. Random Forest Feature Importance

In [ ]:
rf_sel = RandomForestClassifier(
    n_estimators=200, random_state=42, n_jobs=-1,
    class_weight='balanced'  # Manejo de desbalance
)
rf_sel.fit(X_train, y_train)

rf_fi = pd.Series(
    rf_sel.feature_importances_, index=X_train.columns
).sort_values(ascending=False)
rf_fi_pct = (rf_fi / rf_fi.sum() * 100).round(2)

print('📊 TOP 15 RF FEATURE IMPORTANCE:')
for feat, imp in rf_fi_pct.head(15).items():
    bar = '█' * int(imp / 1.5)
    print(f'  {feat:40s}: {imp:5.2f}%  {bar}')

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.RdYlGn(rf_fi.values / rf_fi.max())
rf_fi.plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
ax.set_title('Feature Importance — Random Forest (Churn)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importancia')
for i, (val, feat) in enumerate(zip(rf_fi.values[::-1], rf_fi.index[::-1])):
    ax.text(val + 0.001, i, f'{val*100:.1f}%', va='center', fontsize=7)
plt.tight_layout()
plt.savefig('../data/processed/churn_rf_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Logistic Regression con L1 (LASSO)

In [ ]:
lr_l1 = LogisticRegression(
    penalty='l1', solver='liblinear', C=0.1,
    class_weight='balanced', random_state=42, max_iter=1000
)
lr_l1.fit(X_train_sc, y_train)

lr_coefs = pd.Series(
    np.abs(lr_l1.coef_[0]), index=X_train.columns
).sort_values(ascending=False)

zero_coefs = lr_coefs[lr_coefs == 0].index.tolist()
print(f'🔍 Features con coef=0 (LASSO): {len(zero_coefs)}')
if zero_coefs:
    print(f'   Eliminadas: {zero_coefs}')

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c' if v > 0 else '#bdc3c7' for v in lr_coefs.values]
lr_coefs.plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
ax.set_title('Logistic Regression L1 — Coeficientes Absolutos (Churn)', fontweight='bold')
ax.set_xlabel('|Coeficiente|')
plt.tight_layout()
plt.savefig('../data/processed/churn_lasso_coefs.png', dpi=150)
plt.show()

## 7. Ranking Combinado y Selección Final

In [ ]:
def normalize(s):
    if s.max() == s.min(): return s * 0
    return (s - s.min()) / (s.max() - s.min())

feats = list(X_train.columns)

scores_df = pd.DataFrame({
    'MutualInfo':    normalize(mi_series.reindex(feats).fillna(0)),
    'Chi2':          normalize(chi2_scores['Chi2_Score'].reindex(feats).fillna(0)),
    'RF_Importance': normalize(rf_fi.reindex(feats).fillna(0)),
    'LR_L1':         normalize(lr_coefs.reindex(feats).fillna(0)),
}, index=feats)

scores_df['SCORE_FINAL'] = scores_df.mean(axis=1)
scores_df = scores_df.sort_values('SCORE_FINAL', ascending=False)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
sns.heatmap(scores_df.drop(columns='SCORE_FINAL'), annot=True, fmt='.2f',
            cmap='RdYlGn', ax=axes[0], linewidths=0.5)
axes[0].set_title('Scores Normalizados por Método', fontweight='bold')

scores_df['SCORE_FINAL'].plot(kind='barh', ax=axes[1],
    color=plt.cm.RdYlGn(scores_df['SCORE_FINAL'].values / scores_df['SCORE_FINAL'].max()),
    edgecolor='white')
axes[1].set_title('Score Final Combinado (Churn)', fontweight='bold')
axes[1].axvline(x=0.15, color='red', linestyle='--', alpha=0.7, label='Umbral 0.15')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/churn_feature_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

# Selección
THRESHOLD = 0.10
selected = scores_df[scores_df['SCORE_FINAL'] >= THRESHOLD].index.tolist()

# Garantizar features clave
mandatory = ['tenure', 'MonthlyCharges', 'Contract_Month-to-month', 'is_monthly_contract']
for f in mandatory:
    if f in X_train.columns and f not in selected:
        selected.append(f)

print(f'\n✅ Features seleccionadas ({len(selected)}):')
for f in selected:
    score = scores_df.loc[f, 'SCORE_FINAL'] if f in scores_df.index else 'N/A'
    print(f'  {f:45s}: {score:.3f}' if isinstance(score, float) else f'  {f}')

eliminated = [f for f in feats if f not in selected]
print(f'\n❌ Eliminadas ({len(eliminated)}): {eliminated}')

## 8. Guardado

In [ ]:
# Datasets con features seleccionadas
X_train[selected].to_csv('../data/processed/churn_X_train_sel.csv', index=False)
X_test[selected].to_csv('../data/processed/churn_X_test_sel.csv', index=False)
X_train_sc[selected].to_csv('../data/processed/churn_X_train_sc_sel.csv', index=False)
X_test_sc[selected].to_csv('../data/processed/churn_X_test_sc_sel.csv', index=False)

scores_df.to_csv('../data/processed/churn_feature_scores.csv')

fi['selected_features'] = selected
fi['n_selected'] = len(selected)
fi['eliminated_features'] = eliminated
with open('../data/processed/churn_feature_info.json', 'w') as f:
    json.dump(fi, f, indent=2)

joblib.dump(rf_sel, '../models/churn_rf_selector.pkl')
joblib.dump(selected, '../models/churn_selected_features.pkl')

print('✅ Datasets y artefactos guardados')
print(f'   {len(selected)} features seleccionadas de {len(feats)} totales')

## 9. 📝 Conclusiones

1. **`Contract_Month-to-month`** es la feature más importante en todos los métodos
2. **`tenure`** tiene alta MI y RF importance — clientes nuevos tienen mayor riesgo
3. **`is_monthly_contract`**, **`num_services`** son features derivadas muy útiles
4. **`InternetService_Fiber optic`** muestra alta tasa de churn
5. **`TechSupport_enc`**, **`OnlineSecurity_enc`** discriminan bien entre clases
6. **`gender_enc`** y algunas flags de servicios tienen muy baja importancia → eliminadas
7. El ranking combinado de 4 métodos da selección robusta